Machine Learning implementation:

Step 1: 
    - Transform the dataset:
        - Conducted the BLS and LS analyses to extract features such as period, power, and SDE for each light curve. 
        - These features are stored in dicts, bls_params and ls_params.
        - The features are then compiled into a structured format (e.g., a pandas DataFrame) for machine learning, with gaia id and filter included.
Step 2:
    - Data description:
        - visualize the distribution of the extracted features (e.g., histograms, scatter plots) to understand their characteristics and identify any patterns or correlations.
        - printing summary statistics (mean, median, standard deviation) for each feature to gain insights into their central tendencies and variability.
        - using sns.pairplot to visualize pairwise relationships between features and identify potential clusters or outliers in the data.
Step 3: 
    - Using kmeans method to see the best number of clusters via elbow and silhouette method and scores, ie:
        inertia = []
        silhouette_scores = []
        K_range = range(2, 11)

        for k in K_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = kmeans.fit_predict(X_scaled)
            inertia.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X_scaled, labels))

        # Save elbow plot
        plt.figure(figsize=(8,5))
        plt.plot(list(K_range), inertia, 'bo-')
        plt.xlabel('Number of clusters (K)')
        plt.ylabel('Inertia')
        plt.title('Elbow Method')
        plt.savefig("outputs/elbow_method.png", dpi=150)
        plt.show()

        # Save silhouette plot
        plt.figure(figsize=(8,5))
        plt.plot(list(K_range), silhouette_scores, 'ro-')
        plt.xlabel('Number of clusters (K)')
        plt.ylabel('Silhouette Score')
        plt.title('Silhouette Scores')
        plt.savefig("outputs/silhouette_scores.png", dpi=150)
        plt.show()
Step 4:
    - Using the silhouette scores get our optimal K clusters:
        optimal_k = np.argmax(silhouette_scores) + 2
        print(f"Best K (by silhouette): {optimal_k}")
        kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=50)
        df['Cluster'] = kmeans.fit_predict(X_scaled)
Step 5: 
    - Obtain their cluster centers:
        centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
        centers_df = pd.DataFrame(centers_original, columns=features)
        centers_df.to_csv("outputs/cluster_centers.csv", index=False)
        print("\nCluster Centers (original scale):")
        print(centers_df)
Step 6: 
    - Plot clusters: 
        plt.figure(figsize=(8,6))
        sns.scatterplot(x=X[features[0]], y=X[features[1]], hue=df['Cluster'], palette='Set1', s=100)
        plt.scatter(centers_original[:,0], centers_original[:,1], s=300, c='yellow', marker='X', label='Centroids')
        plt.title('Customer Segmentation')
        plt.legend()
        plt.savefig("outputs/cluster_scatter.png", dpi=150)
        plt.show()
Step 7: 
    - Dimentionality reduction via UMAP or PCA
        example of PCA:
            pca = PCA(n_components=2, random_state=42)
            pca_data = pca.fit_transform(X_scaled)
            df['PCA1'] = pca_data[:, 0]
            df['PCA2'] = pca_data[:, 1]

            plt.figure(figsize=(8,6))
            sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='Cluster', palette='Set2', s=100)
            plt.title('Clusters in PCA Space')
            plt.savefig("outputs/pca_clusters.png", dpi=150)
            plt.show()